In [1]:
%pip install geopandas networkx fiona pyproj shapely

Note: you may need to restart the kernel to use updated packages.


In [2]:
import geopandas as gpd
import pandas as pd
import networkx as nx
import fiona
from shapely.geometry import Point
from collections import Counter


# Uploading files 

In [3]:
roads_filtered = gpd.read_file(
    "/Volumes/LaCie/civil_beat_data/evac_routes/road_network_analysis/qgis/layers/osm_roads_filtered.gpkg",
    layer="osm_roads_filtered"
)

print(len(roads_filtered))
print(roads_filtered.columns.tolist()[:20])
print(roads_filtered.crs)


82580
['full_id', 'osm_id', 'osm_type', 'highway', 'line', 'maxspeed:advisory:forward', 'taxi', 'crossing:markings:colour', 'crossing:continuous', 'phone', 'handrail:center', 'url', 'traffic_signals:vibration', 'access_aisle', 'ramp:wheelchair', 'postal_code', 'flashing_lights', 'destroyed:highway', 'damage:type', 'damage:event:wikidata']
EPSG:4326


In [4]:
roads_exploded = gpd.read_file(
    "/Volumes/LaCie/civil_beat_data/evac_routes/road_network_analysis/qgis/layers/roads_exploded.gpkg",
    layer="roads_exploded"
)

print(len(roads_exploded))
print(roads_exploded.columns.tolist()[:20])
print(roads_exploded.crs)

590355
['full_id', 'osm_id', 'osm_type', 'highway', 'line', 'maxspeed:advisory:forward', 'taxi', 'crossing:markings:colour', 'crossing:continuous', 'phone', 'handrail:center', 'url', 'traffic_signals:vibration', 'access_aisle', 'ramp:wheelchair', 'postal_code', 'flashing_lights', 'destroyed:highway', 'damage:type', 'damage:event:wikidata']
EPSG:4326


In [5]:
road_cut_edges = gpd.read_file(
    "/Volumes/LaCie/civil_beat_data/evac_routes/road_network_analysis/qgis/layers/road_cut_edges.gpkg",
    layer="road_cut_edges"
)

print(len(road_cut_edges))
print(road_cut_edges.columns.tolist()[:20])
print(road_cut_edges.crs)

221134
['cat', 'geometry']
EPSG:4326


In [6]:
# Double checking that we get the right feature count for roads_exploded.

expected_segments = 0

for geom in roads_filtered.geometry:

    if geom is None or geom.is_empty: 
        continue

    if geom.geom_type == "LineString":
        expected_segments += len(geom.coords) - 1

    elif geom.geom_type == "MultiLineString": 
        for part in geom.geoms:
            expected_segments += len(part.coords) - 1

print ("Expected exploded segments:", expected_segments)

Expected exploded segments: 590355


In [7]:
# adding segment_id back in 

road_lookup = { 
    geom.wkb: segment_id 
    for geom, segment_id in zip(
        roads_exploded.geometry, 
        roads_exploded["segment_id"]) 
} 

road_cut_edges ["segment_id"] = road_cut_edges.geometry.apply(
    lambda geom:road_lookup.get(geom.wkb)
)

In [8]:
# checking the match rate

print ("total cut edges:", len(road_cut_edges))
print (
    "matched:", 
    road_cut_edges["segment_id"].notna().sum()
)

print (
    "missing:", 
    road_cut_edges["segment_id"].isna().sum()
)


total cut edges: 221134
matched: 221134
missing: 0


# Building a statewide road network graph 

In [9]:
G = nx.MultiGraph()
for row in roads_exploded.itertuples(): 
        
    geom = row.geometry
    if geom is None or geom.is_empty: 
        continue 

    start = tuple(geom.coords[0])
    end = tuple(geom.coords[-1]) 

    G.add_edge (
        start, 
        end, 
        key=row.segment_id, 
        segment_id=row.segment_id
    )

print("Nodes:", G.number_of_nodes())
print("Edges:", G.number_of_edges())
print(
    "connected components:", 
    nx.number_connected_components(G)
)


Nodes: 562616
Edges: 590355
connected components: 253


# Removing all individual cut edges/bridges and identifying the road that is still connected without them

In [10]:
cut_ids = set (
    road_cut_edges ["segment_id"]
    .dropna()
    .astype(int)
)

G_no_bridges = G.copy()

for u, v, key, data in list (
    G_no_bridges.edges(
        keys=True, 
        data=True
    )
):

    if data["segment_id"] in cut_ids:
        G_no_bridges.remove_edge(
            u, 
            v, 
            key = key
        )

print(
    "edges remaining:", 
    G_no_bridges.number_of_edges()
)

print(
    "components after bridges removed:", 
    nx.number_connected_components(
        G_no_bridges
    )
)

edges remaining: 369221
components after bridges removed: 221387


# Assigning graph node to each unbroken block

This makes a road with no bridges (i.e. unbroken) into one unit. 

In [11]:
node_to_block = {} 
block_node_count = {} 

for block_id, component in enumerate (
    nx.connected_components (
        G_no_bridges
    )
): 

    component = set(component) 

    block_node_count[block_id] = len(
        component 
    )

    for node in component: 
        node_to_block[node] = block_id

print (len(block_node_count))

221387


# Calculate road length behind a bridge/cut edge in miles 

In [12]:
roads_proj = roads_exploded.to_crs(
    "ESRI:102007"
)

roads_exploded["segment_miles"] = ( 
    roads_proj.geometry.length 
    / 1609.344 
)

print ( 
    roads_exploded [
    ["segment_id", "segment_miles"] 
    ].head()
)

print(
    "missing miles:", 
    roads_exploded [
    "segment_miles"
    ].isna().sum()
)

   segment_id  segment_miles
0           1       0.057309
1           2       0.088823
2           3       0.005795
3           4       0.009949
4           5       0.002425
missing miles: 0


# Building a bridge-block graph 

The goal is to turn the complicated road network into a simplier tree-like graph. Shows how bridge-free roads are connected by cut edges.   

In [13]:
### Count internal road segments and mileage in each block

block_segment_count = Counter()
block_road_miles = Counter()

segment_miles_lookup = (
    roads_exploded 
    .set_index ("segment_id")[
         "segment_miles"
    ]
    .to_dict()
)

for u, v, key, data in (
    G_no_bridges.edges (
        keys=True, 
        data=True
    )
):
    block_id = node_to_block[u]

    block_segment_count [
        block_id
    ] += 1 

    block_road_miles [
        block_id
    ] += segment_miles_lookup.get(
        data["segment_id"], 
        0 
    )

In [14]:
### simplified graph

B = nx.Graph()

for block_id in block_node_count:

    B.add_node (
        block_id, 
        node_count=block_node_count[
            block_id
        ] 
    ) 

In [15]:
### connecting bridge-free blocks using the original cut edges 

for row in road_cut_edges.itertuples(): 
    geom = row.geometry 

    u = tuple(geom.coords[0])
    v = tuple(geom.coords[-1])

    block_u = node_to_block[u]
    block_v = node_to_block[v]

    if block_u == block_v: 
        continue

    B.add_edge(
        block_u, 
        block_v, 
        segment_id=int(
            row.segment_id
        )
    )

print("block nodes:", B.number_of_nodes())
print("bridge connections:", B.number_of_edges())
print(
    "block-network components:", 
    nx.number_connected_components(B)
)


block nodes: 221387
bridge connections: 221134
block-network components: 253


In [16]:
### calculating road network behind each bridge, reusable python function

def dependent_measure(
    bridge_graph,
    block_weight
):

    results = {}

    for component_nodes in (
        nx.connected_components(
            bridge_graph
        )
    ):

        tree = bridge_graph.subgraph(
            component_nodes
        )

        if tree.number_of_edges() == 0:
            continue

        root = next(iter(tree.nodes))

        parent = {root: None}
        parent_edge = {}

        order = [root]

        for node in order:

            for neighbor, edge_data in (
                tree[node].items()
            ):

                if neighbor == parent.get(
                    node
                ):
                    continue

                if neighbor in parent:
                    continue

                parent[neighbor] = node

                parent_edge[
                    neighbor
                ] = edge_data[
                    "segment_id"
                ]

                order.append(neighbor)

        subtree = {
            node: block_weight.get(
                node,
                0
            )
            for node in tree.nodes
        }

        for node in reversed(
            order[1:]
        ):

            p = parent[node]

            subtree[p] += subtree[
                node
            ]

        total = subtree[root]

        for node in order[1:]:

            segment_id = (
                parent_edge[node]
            )

            side_a = subtree[node]
            side_b = total - side_a

            results[
                segment_id
            ] = min(
                side_a,
                side_b
            )

    return results

In [17]:
### calculating isolated_nodes

isolated_nodes_by_segment = (
    dependent_measure(
        B,
        block_node_count
    )
)

road_cut_edges["isolated_nodes"] = (
    road_cut_edges["segment_id"]
    .map(
        isolated_nodes_by_segment
    )
)


### calculating isolated_segments

isolated_segments_by_segment = (
    dependent_measure(
        B,
        block_segment_count
    )
)

road_cut_edges["isolated_segments"] = (
    road_cut_edges["segment_id"]
    .map(
        isolated_segments_by_segment
    )
)


### calculating isolated_miles

isolated_miles_by_segment = (
    dependent_measure(
        B,
        block_road_miles
    )
)

road_cut_edges["isolated_miles"] = (
    road_cut_edges["segment_id"]
    .map(
        isolated_miles_by_segment
    )
)

road_cut_edges["isolated_miles"] = (
    road_cut_edges["isolated_miles"]
    .clip(lower=0)
)

print (
    road_cut_edges[
        [ 
            "segment_id", 
            "isolated_nodes", 
            "isolated_segments", 
            "isolated_miles"
        ]
    ].head()
)

print(
    road_cut_edges[
        [ 
            "isolated_nodes", 
            "isolated_segments", 
            "isolated_miles"
        ]
    ].isna().sum()
)
    

   segment_id  isolated_nodes  isolated_segments  isolated_miles
0           1               1                  0             0.0
1           2               1                  0             0.0
2           3               2                  0             0.0
3           4               1                  0             0.0
4           5               2                  0             0.0
isolated_nodes       0
isolated_segments    0
isolated_miles       0
dtype: int64


### exporting these to a layer for qgis

isolated_nodes = road_cut_edges_analyzed_v1

isolated_segments = road_cut_edges_analyzed_v2

isolated_miles = road_cut_edges_analyzed_v3 


In [18]:
#### Creating nodes behind the bridge/cut edge

road_cut_edges_nodes = road_cut_edges[ 
    [
        "segment_id", 
        "isolated_nodes", 
        "geometry"
    ]
].copy()

road_cut_edges_nodes.to_file(
    "/Volumes/LaCie/civil_beat_data/evac_routes/road_network_analysis/qgis/layers/road_cut_edges_nodes.gpkg", 
    layer="road_cut_edges_nodes", 
    driver="GPKG"
)


#### Creating segments behind the bridge/cut edge

road_cut_edges_segments = road_cut_edges[
    [
        "segment_id", 
        "isolated_segments", 
        "geometry"
    ]
].copy()
road_cut_edges_segments.to_file(
        "/Volumes/LaCie/civil_beat_data/evac_routes/road_network_analysis/qgis/layers/road_cut_edges_segments.gpkg",
        layer="road_cut_edges_segments", 
        driver="GPKG"
)


#### Creating miles behind the bridge/cut edge

road_cut_edges_miles = road_cut_edges [
    [
        "segment_id", 
        "isolated_miles", 
        "geometry"
    ]
].copy()
road_cut_edges_miles.to_file(
        "/Volumes/LaCie/civil_beat_data/evac_routes/road_network_analysis/qgis/layers/road_cut_edges_miles.gpkg", 
        layer="road_cut_edges_miles", 
        driver="GPKG"
)



# Identifying major road corridors & mileage that gets isolated

These are roads that are more likely to be regional or intercommunity throughways.

In [19]:
major_classes = [
    "motorway", 
    "motorway_link", 
    "trunk", 
    "trunk_link", 
    "primary", 
    "primary_link", 
    "secondary",
    "secondary_link", 
    "tertiary", 
    "tertiary_link"
]

major_roads = roads_exploded [
    roads_exploded["highway"].isin(
        major_classes 
    )
].copy()

print(len(major_roads))

print(
    major_roads[
        "highway"
    ].value_counts()

)

112609
highway
tertiary          61043
primary           16540
secondary         15752
motorway_link      4098
trunk              3873
primary_link       3231
motorway           2628
tertiary_link      2365
secondary_link     1638
trunk_link         1441
Name: count, dtype: int64


In [20]:
### Major corridor statewide road graph 

G_major_roads = nx.MultiGraph()

for row in major_roads.itertuples():

    geom = row.geometry

    if geom is None or geom.is_empty:
        continue

    start = tuple(geom.coords[0])
    end = tuple(geom.coords[-1])

    G_major_roads.add_edge(
        start,
        end,
        key=row.segment_id,
        segment_id=row.segment_id,
        highway=row.highway
    )

#### checking component sizes

major_component_sizes = sorted(
    [
        len(c)
        for c in 
        nx.connected_components(
            G_major_roads
        )
    ],
    reverse=True
)

print(
    major_component_sizes[:20]
)

[51795, 22665, 22088, 8790, 3106, 1094, 132, 54, 53, 13, 13, 6, 4]


In [21]:
### create major_road_components 

major_node_component = {}

for comp_id, component in enumerate(
    nx.connected_components(
        G_major_roads
    )
):

    for node in component:
            major_node_component[
                node 
        ] = comp_id


#### assigning each road segment a component based on its start node

def get_major_component(geom):
    node = tuple(
        geom.coords[0]
    )

    return major_node_component.get(
        node
    )

major_roads[
    "major_comp"
] = major_roads.geometry.apply(
    get_major_component
) 


In [22]:
### creating a layer for QGIS for major road components 

major_roads.to_file(
    "/Volumes/LaCie/civil_beat_data/evac_routes/road_network_analysis/qgis/layers/major_road_components.gpkg",
    layer="major_road_components",
    driver="GPKG"
)

In [23]:
### Identifying individual bridges in the major road network

major_bridges = list(
    nx.bridges(
        nx.Graph(G_major_roads)
    )
)

candidate_pairs = set(
    major_bridges
)

### excluding node pairs with parallel edges in the major roads graph

major_bridge_segment_ids = []

for u, v, key, data in (
    G_major_roads.edges(
        keys=True, 
        data=True
    )
):

    pair = (
        (u,v)
        if (u,v)
        in candidate_pairs
        else (v,u)
    )

    if pair in candidate_pairs:
        if (
            G_major_roads.number_of_edges(
                u, 
                v
            ) 
            == 1
        ):
            major_bridge_segment_ids.append(
                data["segment_id"]
            )


### creating the layer 

major_cut_edges = major_roads[
    major_roads[
        "segment_id"
        ].isin(
            major_bridge_segment_ids
    )

].copy()

### exporting it to major_cut_edges

major_cut_edges.to_file(
    "/Volumes/LaCie/civil_beat_data/evac_routes/road_network_analysis/qgis/layers/major_corridor_cut_edges.gpkg", 
    layer="major_corridor_cut_edges", 
    driver="GPKG"
)


In [24]:
### putting major vulnerable corridors into longer connected sections

H = nx.Graph()

for row in major_cut_edges.itertuples():
    geom = row.geometry

    u = tuple(
        geom.coords[0]
    )

    v = tuple(
        geom.coords[-1]
    )

    H.add_edge(
        u,
        v,
        segment_id=row.segment_id
    )

#### adding corridor ID 
segment_to_vulnerable_corridor = {}

for corridor_id, component in enumerate(
    nx.connected_components(H)
):

    subgraph = H.subgraph(
        component
    )

    for u, v, data in (
        subgraph.edges(
            data=True
        )
    ):

        segment_to_vulnerable_corridor[
            data["segment_id"]
        ]=corridor_id

#### attaching to the road layer 

major_cut_edges[
    "corridor_id"
] = (
    major_cut_edges[
        "segment_id"
    ].map(
        segment_to_vulnerable_corridor
    )
)



In [25]:
### grouping major road segments into larger corridor sections

junction_nodes = {
    node
    for node in G_major_roads.nodes ()
    if len(
        set(
            G_major_roads.neighbors(
                node
            )
        )
    ) !=2
}

print (
    "meaningful junction nodes:", 
    len(junction_nodes)
)
    

meaningful junction nodes: 5358


# diagnosing a problem with the too low meaningful junction nodes

this was caused by an indent error on the creation of G_major_roads, but I fixed it.

In [26]:
print("Nodes:", G_major_roads.number_of_nodes())
print("Edges:", G_major_roads.number_of_edges())
print(
    "Connected components:",
    nx.number_connected_components(G_major_roads)
)

Nodes: 109813
Edges: 112609
Connected components: 13


In [27]:
from collections import Counter

neighbor_counts = Counter(
    len(set(G_major_roads.neighbors(node)))
    for node in G_major_roads.nodes()
)

print(neighbor_counts)

Counter({2: 104455, 3: 3739, 4: 1036, 1: 494, 5: 81, 6: 8})


In [28]:
print("Major road features:", len(major_roads))
print(major_roads["highway"].value_counts())
print("graph edges:", G_major_roads.number_of_edges())

Major road features: 112609
highway
tertiary          61043
primary           16540
secondary         15752
motorway_link      4098
trunk              3873
primary_link       3231
motorway           2628
tertiary_link      2365
secondary_link     1638
trunk_link         1441
Name: count, dtype: int64
graph edges: 112609


In [29]:
### grouping continuous major road segments into corridor sections

visited_segment_ids = set()
corridor_sections = []

section_id = 1

for start in junction_nodes:

    for neighbor in G_major_roads.neighbors(start):

        edge_dict = G_major_roads.get_edge_data(
            start,
            neighbor
        )

        for key, edge_data in edge_dict.items():

            segment_id = edge_data[
                "segment_id"
            ]

            if segment_id in visited_segment_ids:
                continue

            segment_ids = [
                segment_id
            ]

            visited_segment_ids.add(
                segment_id
            )

            previous = start
            current = neighbor

            while current not in junction_nodes:

                next_neighbors = [
                    n
                    for n in set(
                        G_major_roads.neighbors(
                            current
                        )
                    )
                    if n != previous
                ]

                if len(next_neighbors) != 1:
                    break

                next_node = next_neighbors[0]

                next_edges = (
                    G_major_roads.get_edge_data(
                        current,
                        next_node
                    )
                )

                unvisited = [
                    data["segment_id"]
                    for key, data
                    in next_edges.items()
                    if data["segment_id"]
                    not in visited_segment_ids
                ]

                if len(unvisited) == 0:
                    break

                for sid in unvisited:

                    segment_ids.append(
                        sid
                    )

                    visited_segment_ids.add(
                        sid
                    )

                previous = current
                current = next_node

            corridor_sections.append({
                "section_id": section_id,
                "start_node": start,
                "end_node": current,
                "segment_ids": segment_ids,
                "segment_count": len(segment_ids)
            })

            section_id += 1


print(
    "Corridor sections:",
    len(corridor_sections)
)

Corridor sections: 8154


In [30]:
### quick gut check 

print(
    "Visited major-road segments:",
    len(visited_segment_ids)
)

print(
    "Total major-road segments:",
    len(major_roads)
)

print(
    "Corridor sections:",
    len(corridor_sections)
)

Visited major-road segments: 112609
Total major-road segments: 112609
Corridor sections: 8154


In [31]:
### inspect corridor section sizes

sections_df = pd.DataFrame(
    corridor_sections
)

print(
    sections_df[
        "segment_count"
    ].describe()
)


count    8154.000000
mean       13.810277
std        77.331639
min         1.000000
25%         2.000000
50%         5.000000
75%        11.000000
max      4711.000000
Name: segment_count, dtype: float64


In [32]:
### simplified corridor graph 

G_corridor = nx.MultiGraph()

for section in corridor_sections:
    G_corridor.add_edge(
        section["start_node"],
        section["end_node"], 
        key=section[
            "section_id"
        ],
        section_id=section[
            "section_id"
        ],
        segment_count=section[
            "segment_count"
        ]
    )

print(
    "corridor graph nodes:", 
    G_corridor.number_of_nodes()
)

print(
    "connected components:", 
    nx.number_connected_components(
        G_corridor
    )
)


corridor graph nodes: 5358
connected components: 13


In [34]:
### identify single-corridor dependencies

G_corridor_simple = nx.Graph(
    G_corridor
)

candidate_bridge_pairs = set(
    nx.bridges(
        G_corridor_simple
    )
)

#### making sure parallel corridor edges are not falsely classified as bridges

corridor_bridge_ids = []

for u, v, key, data in (
    G_corridor.edges(
        keys=True,
        data=True
    )
):

    pair = (
        (u, v)
        if (u, v) in candidate_bridge_pairs
        else (v, u)
    )

    if pair in candidate_bridge_pairs:

        if (
            G_corridor.number_of_edges(
                u,
                v
            )
            == 1
        ):

            corridor_bridge_ids.append(
                data["section_id"]
            )


print(
    "Single-corridor-dependent sections:",
    len(corridor_bridge_ids)
)

print(
    "Percent:",
    round(
        len(corridor_bridge_ids)
        / G_corridor.number_of_edges()
        * 100,
        1
    )
)

Single-corridor-dependent sections: 597
Percent: 7.3


In [37]:
### calculating how much major corridor is cut off if bridges removed

major_roads_proj = (
    major_roads.to_crs(
        "ESRI:102007"
    )
)

major_roads["segment_miles"] = (
    major_roads_proj.geometry.length
    / 1609.344
)


#### calculating the mileage for each corridor section

section_miles = {}

for section in corridor_sections:

    ids = section[
        "segment_ids"
    ]

    miles = major_roads.loc[
        major_roads[
            "segment_id"
        ].isin(ids),
        "segment_miles"
    ].sum()

    section_miles[
        section["section_id"]
    ] = miles


sections_df[
    "section_miles"
] = (
    sections_df[
        "section_id"
    ].map(section_miles)
)


#### adding mileage to the corridor graph

for u, v, key, data in (
    G_corridor.edges(
        keys=True,
        data=True
    )
):

    section_id = data[
        "section_id"
    ]

    data["miles"] = float(
        section_miles[
            section_id
        ]
    )


In [43]:
### measuring the major corridor network behind each corridor bridge

corridor_dependency_size = {}
dependent_corridor_miles = {}

for section_id in corridor_bridge_ids: 
    section = next(
        s 
        for s in corridor_sections
        if s[
            "section_id"
        ] == section_id
    )

    u = section[
        "start_node"
    ]

    v = setion[
        "end_node" 
    ]

    G_temp = (
        G_corridor.copy()
    )

    G_temp.remove_edge(
        u,
        v,
        key=section_id
    )

    comp_u = (
        nx.node_connected_component(
            G_temp, 
            u
        )
    )

    comp_v = (
        nx.node_connected_component(
            G_temp, 
            v
        )
    )

    edges_u = (
        G_temp.subgraph(
            comp_u
        ).number_of_edges()
    )

    edges_v = (
        G_temp.subgraph(
            comp_v
        ).number_of_edges()
    )

    corridor_dependency_size[
        section_id
    ] = min(
        edges_u, 
        edges_v
    )

    miles_u = sum(
        data["miles"] 
        for _, _, _, data in 
        G_temp.subgraph(
            comp_u
        ).edges(
            keys=True, 
            data=True 
        )
    )

    miles_v = sum(
        data["miles"]
        for _, _, _, data in 
        G_temp.subgraph(
            comp_v
        ).edges(
            keys=True,
            data=True
        )
    )
    
    dependent_corridor_miles[
        section_id
    ] = min(
        miles_u, 
        miles_v
    )


In [56]:
### create layer of major roads that are the bridges that if removed, sever the rest of the (major?) network & how much road is isolated 

bridge_section_ids = set(
    corridor_bridge_ids
)

segment_to_section = {}

for section in corridor_sections:
    if (
        section["section_id"]
        in bridge_section_ids
    ):
        for segment_id in (
            section[
            "segment_ids"
            ]
        ):
            segment_to_section[
                segment_id
            ] = section[
                "section_id"
            ]
#### selecting road geometries

corridor_bridges_map = ( 
    major_roads[
        major_roads[
            "segment_id"
        ].isin(
            segment_to_section.keys()
        )
    ].copy()
)

corridor_bridges_map[
    "section_id"
] = (
    corridor_bridges_map[
        "segment_id"
    ].map(
        segment_to_section
    )
)

#### attaching dependency measures

corridor_bridges_map[
    "dependent_corridor_sections"
] = (
    corridor_bridges_map[
        "section_id"
    ].map(
        corridor_dependency_size
    )
)

corridor_bridges_map[
    "dependent_major_miles"
] = (
    corridor_bridges_map[
        "section_id"
    ].map(
        dependent_corridor_miles
    )
)

#### exporting to a layer for QGIS 

corridor_bridges_map.to_file(
    "/Volumes/LaCie/civil_beat_data/evac_routes/road_network_analysis/qgis/layers/single_major_corridor_dependency.gpkg", 
    layer = "single_major_corridor_dependency", 
    driver="GPKG"
)


In [57]:
### Building the roads that are cut off if a major-road bridge is removed
#### creating dependent_major_roads_by_corridor

dependent_records = []

for section_id in corridor_bridge_ids:

    section = next(
        s
        for s in corridor_sections
        if s["section_id"] == section_id
    )

    u = section["start_node"]
    v = section["end_node"]

    G_temp = G_corridor.copy()

    G_temp.remove_edge(
        u,
        v,
        key=section_id
    )

    comp_u = nx.node_connected_component(
        G_temp,
        u
    )

    comp_v = nx.node_connected_component(
        G_temp,
        v
    )

    miles_u = sum(
        data["miles"]
        for _, _, _, data in
        G_temp.subgraph(
            comp_u
        ).edges(
            keys=True,
            data=True
        )
    )

    miles_v = sum(
        data["miles"]
        for _, _, _, data in
        G_temp.subgraph(
            comp_v
        ).edges(
            keys=True,
            data=True
        )
    )

    #### choose the smaller side by major-road mileage

    if miles_u <= miles_v:
        dependent_nodes = comp_u
    else:
        dependent_nodes = comp_v

    dependent_subgraph = G_temp.subgraph(
        dependent_nodes
    )

    dependent_section_ids = {
        data["section_id"]
        for _, _, _, data in
        dependent_subgraph.edges(
            keys=True,
            data=True
        )
    }

    for dependent_section_id in dependent_section_ids:

        dependent_records.append({
            "trigger_section_id": section_id,
            "dependent_section_id": dependent_section_id
        })


#### Convert dependent corridor sections to original major-road segment IDs

dependent_sections_df = pd.DataFrame(
    dependent_records
)

section_to_segments = {
    section["section_id"]:
        section["segment_ids"]
    for section in corridor_sections
}

road_dependency_records = []

for row in dependent_sections_df.itertuples():

    segment_ids = section_to_segments[
        row.dependent_section_id
    ]

    for segment_id in segment_ids:

        road_dependency_records.append({
            "trigger_section_id":
                row.trigger_section_id,
            "dependent_section_id":
                row.dependent_section_id,
            "segment_id":
                segment_id
        })


#### Join road geometry back onto those segment IDs

road_dependency_df = pd.DataFrame(
    road_dependency_records
)

dependent_roads_map = road_dependency_df.merge(
    major_roads,
    on="segment_id",
    how="left"
)

dependent_roads_map = gpd.GeoDataFrame(
    dependent_roads_map,
    geometry="geometry",
    crs=major_roads.crs
)


#### Export to QGIS

dependent_roads_map.to_file(
    "/Volumes/LaCie/civil_beat_data/evac_routes/road_network_analysis/qgis/layers/dependent_major_roads_by_corridor.gpkg",
    layer="dependent_major_roads_by_corridor",
    driver="GPKG"
)

In [58]:
#### quick check. we want missing geometries to be 0

print("Dependent section records:", len(dependent_records))
print("Dependent road records:", len(road_dependency_records))
print("Mapped road features:", len(dependent_roads_map))

print(
    "Missing geometries:",
    dependent_roads_map.geometry.isna().sum()
)

Dependent section records: 2259
Dependent road records: 68926
Mapped road features: 68926
Missing geometries: 0


# Calculating areas that have no alternative route at all

Previously, we've only calculated areas with no route via a major road. But neighborhood roads are proving to be a key option. So we need to calculate the areas that if a bridge is removed, they will be completely isolated with no alternative. This will give us areas that are truly one way in one way out.


In [64]:
### rebuilding the complete full road graph. Shouldn't have to recreate G, but add here if need be.

#### creating an edge lookup

edge_lookup = {}

for u, v, key, data in (
    G.edges(
        keys=True,
        data=True
    )
):

    edge_lookup[
        data["segment_id"]
    ] = (
        u,
        v,
        key,
        data.copy()
    )

print(
    "Edges indexed:",
    len(edge_lookup)
)


Edges indexed: 590355


In [65]:
### Testing corridor dependencies against all drivable roads 

full_network_alternative = {}

for section_id in (
    corridor_bridge_ids
):

    section = next(
        s
        for s in corridor_sections
        if s[
            "section_id"
        ] == section_id
    )

    start_node = section[
        "start_node"
    ]

    end_node = section[
        "end_node"
    ]

    removed_edges = []

    for segment_id in (
        section["segment_ids"]
    ):

        if segment_id in edge_lookup:

            u, v, key, data = (
                edge_lookup[
                    segment_id
                ]
            )

            if G.has_edge(
                u,
                v,
                key
            ):

                G.remove_edge(
                    u,
                    v,
                    key=key
                )

                removed_edges.append(
                    (
                        u,
                        v,
                        key,
                        data
                    )
                )

    full_network_alternative[
        section_id
    ] = nx.has_path(
        G,
        start_node,
        end_node
    )

    for u, v, key, data in (
        removed_edges
    ):

        G.add_edge(
            u,
            v,
            key=key,
            **data
        )

alt_results = pd.Series(
    full_network_alternative
)

print(
    alt_results.value_counts()
)


True     409
False    188
Name: count, dtype: int64


In [67]:
### Create a test to see how many corridor dependencies have an alternative route on a drivable part of the network that just isn't a major road.

corridor_bridges_map[
    "full_network_alternative"
] = (
    corridor_bridges_map[
        "section_id"
    ].map(
        full_network_alternative
    )
) 

print(
    corridor_bridges_map[
        [
            "section_id",
            "full_network_alternative"
        ]
    ]
    .drop_duplicates(
        "section_id"
    )[
        "full_network_alternative"
    ]
    .value_counts()
)


corridor_bridges_map.to_file(
    "/Volumes/LaCie/civil_beat_data/evac_routes/road_network_analysis/qgis/layers/corridor_dependency_full_network_test.gpkg", 
    layer = "corridor_dependency_full_network_test", 
    driver="GPKG"
)   
    

full_network_alternative
True     409
False    188
Name: count, dtype: int64


In [70]:
### Building the entire road network dependent on the “no alternative” corridors

no_alt_section_ids = [
    section_id
    for section_id,
        has_alternative
    in full_network_alternative.items()
    if has_alternative is False
]

print(
    "No-alternative corridor sections:",
    len(no_alt_section_ids)
)

No-alternative corridor sections: 188


In [71]:
### identifying dependent side of no-alternative corridor. 


dependent_full_records = []

total = len(
    no_alt_section_ids
)

for i, section_id in enumerate(
    no_alt_section_ids,
    start=1
):

    section = next(
        s
        for s in corridor_sections
        if s[
            "section_id"
        ] == section_id
    )

    start_node = section[
        "start_node"
    ]

    end_node = section[
        "end_node"
    ]

    removed_edges = []

    #### Temporarily remove trigger corridor
    
    for segment_id in (
        section["segment_ids"]
    ):

        if segment_id in edge_lookup:

            u, v, key, data = (
                edge_lookup[
                    segment_id
                ]
            )

            if G.has_edge(
                u,
                v,
                key
            ):

                G.remove_edge(
                    u,
                    v,
                    key=key
                )

                removed_edges.append(
                    (
                        u,
                        v,
                        key,
                        data
                    )
                )

    #### Find the two disconnected sides
    comp_start = (
        nx.node_connected_component(
            G,
            start_node
        )
    )

    comp_end = (
        nx.node_connected_component(
            G,
            end_node
        )
    )

    #### Smaller side by node count = dependent.
    if (
        len(comp_start)
        <= len(comp_end)
    ):

        dependent_nodes = (
            comp_start
        )

    else:

        dependent_nodes = (
            comp_end
        )

    seen_segments = set()

    #### Record every road edge that is entirely inside the component 

    for u in dependent_nodes:

        for v, keydict in (
            G[u].items()
        ):

            if (
                v
                not in dependent_nodes
            ):
                continue

            for key, data in (
                keydict.items()
            ):

                segment_id = (
                    data[
                        "segment_id"
                    ]
                )

                if (
                    segment_id
                    not in
                    seen_segments
                ):

                    dependent_full_records.append({
                        "trigger_section_id":
                            section_id,
                        "segment_id":
                            segment_id
                    })

                    seen_segments.add(
                        segment_id
                    )

    #### Restore the corridor before testing the next one
   
    for u, v, key, data in (
        removed_edges
    ):

        G.add_edge(
            u,
            v,
            key=key,
            **data
        )

    if (
        i % 10 == 0
        or i == total
    ):

        print(
            f"Completed {i} of {total}"
        )

Completed 10 of 188
Completed 20 of 188
Completed 30 of 188
Completed 40 of 188
Completed 50 of 188
Completed 60 of 188
Completed 70 of 188
Completed 80 of 188
Completed 90 of 188
Completed 100 of 188
Completed 110 of 188
Completed 120 of 188
Completed 130 of 188
Completed 140 of 188
Completed 150 of 188
Completed 160 of 188
Completed 170 of 188
Completed 180 of 188
Completed 188 of 188


In [87]:
### creating a QGIS layer for dependent roads with no alternative

dependent_full_df = (
    pd.DataFrame(
        dependent_full_records
    )
)

print(
    "Dependent road records:",
    len(dependent_full_df)
)


#### merging with full road layer

dependent_full_roads = (
    dependent_full_df.merge(
        roads_exploded,
        on="segment_id",
        how="left"
    )
)

dependent_full_roads = (
    gpd.GeoDataFrame(
        dependent_full_roads,
        geometry="geometry",
        crs=roads_exploded.crs
    )
)

print(
    "Rows:",
    len(
        dependent_full_roads
    )
)

print(
    "Missing geometries:",
    dependent_full_roads.geometry.isna().sum()
)

#### export 

dependent_full_roads.to_file(
    "/Volumes/LaCie/civil_beat_data/evac_routes/road_network_analysis/qgis/layers/dependent_full_road_network.gpkg",
    layer="dependent_full_road_network",
    driver="GPKG"
)

Dependent road records: 130324
Rows: 130324
Missing geometries: 0


# Some double checking & examining results 

In [73]:
dependent_full_df = pd.DataFrame(
    dependent_full_records
)

print(
    dependent_full_df.head()
)

print(
    "Unique trigger sections:",
    dependent_full_df[
        "trigger_section_id"
    ].nunique()
)

print(
    "Unique dependent road segments:",
    dependent_full_df[
        "segment_id"
    ].nunique()
)

   trigger_section_id  segment_id
0                  32      332238
1                  32      332239
2                  32      363313
3                  32      363314
4                  32      345713
Unique trigger sections: 124
Unique dependent road segments: 64748


The above gave a different number of corridors that if removed, leave dependent roads without any alternative. we'd expect 188, but we get 124 "unique trigger sections." so in the next code cell, let's test what might be going on.

In [75]:
#### testing different unique trigger sections

diagnostic_records = []

for section_id in no_alt_section_ids:

    section = next(
        s
        for s in corridor_sections
        if s["section_id"] == section_id
    )

    start_node = section["start_node"]
    end_node = section["end_node"]

    removed_edges = []

    # Remove trigger corridor
    for segment_id in section["segment_ids"]:

        if segment_id in edge_lookup:

            u, v, key, data = edge_lookup[
                segment_id
            ]

            if G.has_edge(
                u,
                v,
                key
            ):

                G.remove_edge(
                    u,
                    v,
                    key=key
                )

                removed_edges.append(
                    (
                        u,
                        v,
                        key,
                        data
                    )
                )

    comp_start = nx.node_connected_component(
        G,
        start_node
    )

    comp_end = nx.node_connected_component(
        G,
        end_node
    )

    # Number of edges on each side
    edges_start = G.subgraph(
        comp_start
    ).number_of_edges()

    edges_end = G.subgraph(
        comp_end
    ).number_of_edges()

    # Smaller side by node count,
    # matching your current method
    if len(comp_start) <= len(comp_end):

        dependent_nodes = comp_start
        dependent_edges = edges_start

    else:

        dependent_nodes = comp_end
        dependent_edges = edges_end

    diagnostic_records.append({
        "section_id": section_id,
        "start_nodes": len(comp_start),
        "end_nodes": len(comp_end),
        "start_edges": edges_start,
        "end_edges": edges_end,
        "dependent_nodes": len(
            dependent_nodes
        ),
        "dependent_edges": dependent_edges
    })

    # Restore trigger corridor
    for u, v, key, data in removed_edges:

        G.add_edge(
            u,
            v,
            key=key,
            **data
        )


diagnostics = pd.DataFrame(
    diagnostic_records
)

print(
    "Total no-alternative corridors:",
    len(diagnostics)
)

print(
    "Dependent side has zero roads:",
    (
        diagnostics[
            "dependent_edges"
        ] == 0
    ).sum()
)

print(
    "Dependent side has roads:",
    (
        diagnostics[
            "dependent_edges"
        ] > 0
    ).sum()
)

Total no-alternative corridors: 188
Dependent side has zero roads: 64
Dependent side has roads: 124


In [76]:
#### also want to double check the number of dependency records, because it's larger than expected. 

trigger_counts = (
    dependent_full_df
    .groupby("trigger_section_id")
    ["segment_id"]
    .nunique()
    .sort_values(
        ascending=False
    )
)

print(trigger_counts.head(20))

print(
    trigger_counts.describe()
)

print(
    "Total records:",
    len(dependent_full_df)
)

print(
    "Unique trigger/segment pairs:",
    len(
        dependent_full_df[
            [
                "trigger_section_id",
                "segment_id"
            ]
        ].drop_duplicates()
    )
)

print(
    "Unique road segments overall:",
    dependent_full_df[
        "segment_id"
    ].nunique()
)

trigger_section_id
1945    24000
1614    23487
2140     9956
7819     7725
4629     6509
5687     6403
6790     5692
3296     4129
5903     3824
3577     2467
2169     1645
2014     1602
4281     1566
1393     1464
1701     1429
2138     1423
1155     1400
6158     1326
4828     1196
371      1115
Name: segment_id, dtype: int64
count      124.000000
mean      1051.000000
std       3296.110566
min          1.000000
25%         26.750000
50%        144.500000
75%        648.500000
max      24000.000000
Name: segment_id, dtype: float64
Total records: 130324
Unique trigger/segment pairs: 130324
Unique road segments overall: 64748


The segment above is to learn more about the Total dependency records: 130324. We learn that 130,324 trigger road relationships actually only represent 64,748 distinct road segments becasue there's overlap where road segments appear under more than one trigger corridor and dependencies are nested. Basically there's one trigger here and then another further down and so on... some of those trigger corridors have large dependent road networks. we could look at this further. 

In [77]:
#### double checking the 64 missing "trigger corridors" i.e. bridge/edge cuts major roads are actually just the end of the road, there's no more road beyond it.

print(
    "Total no-alternative corridors:",
    len(diagnostics)
)

print(
    "Dependent side has zero roads:",
    (diagnostics["dependent_edges"] == 0).sum()
)

print(
    "Dependent side has roads:",
    (diagnostics["dependent_edges"] > 0).sum()
)

Total no-alternative corridors: 188
Dependent side has zero roads: 64
Dependent side has roads: 124


In [80]:
#### inspecting the biggest "trigger corridors" sections of the major road network that are bridges/cut edges. 

for test_id in [1945, 1614, 2140, 7819]:

    row = diagnostics[
        diagnostics["section_id"] == test_id
    ]

    print("\nSECTION", test_id)
    print(row.to_string(index=False))


SECTION 1945
 section_id  start_nodes  end_nodes  start_edges  end_edges  dependent_nodes  dependent_edges
       1945        23499      34627        24000      35904            23499            24000

SECTION 1614
 section_id  start_nodes  end_nodes  start_edges  end_edges  dependent_nodes  dependent_edges
       1614        35150      22993        36435      23487            22993            23487

SECTION 2140
 section_id  start_nodes  end_nodes  start_edges  end_edges  dependent_nodes  dependent_edges
       2140         9751      42886         9956      44395             9751             9956

SECTION 7819
 section_id  start_nodes  end_nodes  start_edges  end_edges  dependent_nodes  dependent_edges
       7819        50633       7502        52186       7725             7502             7725


In [88]:
### exporting the top 10 "trigger corridors" i.e. major roads that are bridges or cut edges and if removed sever the dependent area from there rest of the network with no alternative on the full road network

top_10_triggers = (
    trigger_counts
    .head(10)
    .index
    .tolist()
)

print(top_10_triggers)

top_10_dependent = dependent_full_df[
    dependent_full_df["trigger_section_id"].isin(
        top_10_triggers
    )
].copy()

top_10_dependent_map = top_10_dependent.merge(
    roads_exploded,
    on="segment_id",
    how="left"
)

top_10_dependent_map = gpd.GeoDataFrame(
    top_10_dependent_map,
    geometry="geometry",
    crs=roads_exploded.crs
)

top_10_dependent_map.to_file(
    "/Volumes/LaCie/civil_beat_data/evac_routes/road_network_analysis/qgis/layers//top_10_dependent_full_road_network.gpkg",
    layer="top_10_dependent_full_road_network",
    driver="GPKG"
)

[1945, 1614, 2140, 7819, 4629, 5687, 6790, 3296, 5903, 3577]
